# 16.2 — BCG mouse → v08 IID Pearson gene axis

**Superseded for the BCG marathon.** `e22_batch_correction.ipynb` changed the
batch-corrected object we pursue. Use **16.3** (`16.3_bcg_attempts_v08_iid_pearson.ipynb`)
on the August scANVI files. This notebook still maps the **May**
`bcg_mouse_aligned_051026.h5ad` mentor `.X` (old CP10k-style decode) onto v08.

Notebook **16** projected BCG onto the **atlas-full v07** Pearson / Seurat lists
(`hvg_{flavor}_atlas_full_v07.h5ad`). Those are **not** the gene axis of the
current deployment models.

This notebook remaps the same BCG mouse object onto the **uncapped v08 Pearson**
1,000-gene list used by `uncapped_v08_iid` (and by `uncapped_v08` — same `.h5ad`,
different train split):

`cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_a_uncapped_v08.h5ad`

That list shares only **721 / 1000** genes with the v07 Pearson axis, so a v07-aligned
matrix cannot be fed to a v08 checkpoint.

Same design as notebook 16:
- keep the mentor scANVI `.X` (no count swap, no `log1p`)
- map `bcg.var['gene_ids']` (ENSMUSG) → human ENSG via the cached one-to-one ortholog table
- project onto the model gene order; **zero-fill** genes with no BCG column
- report how many of the 1,000 v08 Pearson genes are actually present in BCG

This is **gene-axis alignment only**. `predict_new_input.sh --model-set uncapped_v08_iid`
still wants **raw counts** and will redo projection + Scanpy/`log1p` itself. Use this
notebook to inspect coverage and to produce a v08-aligned copy of the mentor `.X`.

**Does not overwrite** notebook 16 outputs.


In [1]:
import os
import sys

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")
from speciesot_helpers import strip_ensembl_gene_id

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
BCG_PATH = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_051026.h5ad"
DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
ORTHO_CACHE = os.path.join(BASE_DIR, "scripts/.biomart_ortholog_cache.csv")
OUT_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/bcg_mouse_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

V08_PEARSON = os.path.join(DATASET_DIR, "hvg_pearson_residuals_a_uncapped_v08.h5ad")
V07_PEARSON = os.path.join(DATASET_DIR, "hvg_pearson_residuals_atlas_full_v07.h5ad")
V08_MIXHVG = os.path.join(DATASET_DIR, "hvg_mixhvg_a_uncapped_v08.h5ad")
ALIGNED_OUT = os.path.join(DATASET_DIR, "bcg_mouse_aligned_pearson_residuals_uncapped_v08_iid.h5ad")
COVERAGE_CSV = os.path.join(OUT_DIR, "bcg_v08_iid_pearson_hvg_coverage.csv")
MISSING_CSV = os.path.join(OUT_DIR, "bcg_missing_hvg_v08_iid_pearson.csv")

print("BCG source:", BCG_PATH, "exists?", os.path.exists(BCG_PATH))
print("v08 Pearson axis:", V08_PEARSON, "exists?", os.path.exists(V08_PEARSON))
print("v07 Pearson axis:", V07_PEARSON, "exists?", os.path.exists(V07_PEARSON))
print("ortholog cache:", ORTHO_CACHE, "exists?", os.path.exists(ORTHO_CACHE))


BCG source: /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_051026.h5ad exists? True
v08 Pearson axis: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_a_uncapped_v08.h5ad exists? True
v07 Pearson axis: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_atlas_full_v07.h5ad exists? True
ortholog cache: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv exists? True


## 1. Load BCG mouse (keep mentor `.X`)

Same source object as notebook 16. `.X` is scANVI `get_normalized_expression`
(already posed toward the atlas batch). We do **not** replace it with `layers['counts']`.


In [2]:
bcg = sc.read_h5ad(BCG_PATH)
print("BCG read:", bcg.shape, "vars sample:", list(bcg.var_names[:5]))
print("obs cols:", list(bcg.obs.columns))
print("layers:", list(bcg.layers.keys()) if bcg.layers else "(none)")
assert "gene_ids" in bcg.var.columns, "expected BCG .var['gene_ids'] with ENSMUSG"
print("  gene_ids sample:", dict(zip(bcg.var_names[:3], bcg.var["gene_ids"].astype(str).head(3))))

if sp_sparse.issparse(bcg.X):
    bcg.X = bcg.X.astype(np.float32)
else:
    bcg.X = np.asarray(bcg.X, dtype=np.float32)

bcg.obs_names_make_unique()
bcg.obs["condition"] = "mouse"
bcg.obs["species"] = "mouse"
print("cell_type counts:", bcg.obs["cell_type"].value_counts().to_dict())


BCG read: (1406, 10866) vars sample: ['Mrpl15', 'Lypla1', 'Tcea1', 'Atp6v1h', 'Rb1cc1']
obs cols: ['n_genes', 'leiden', 'cell_type', 'study', 'cell_type_original', 'cell_type_scanvi']
layers: ['counts']
  gene_ids sample: {'Mrpl15': 'ENSMUSG00000033845', 'Lypla1': 'ENSMUSG00000025903', 'Tcea1': 'ENSMUSG00000033813'}
cell_type counts: {'LT-HSC treated': 994, 'LT-HSC': 412}


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## 2. Ortholog join: BCG `gene_ids` → human ENSG

One-to-one human↔mouse Ensembl pairs from the notebook 15 BioMart cache.
No symbol lookup — BCG already carries ENSMUSG in `var['gene_ids']`.


In [3]:
assert os.path.exists(ORTHO_CACHE), f"missing ortholog cache: {ORTHO_CACHE}"
ortho_df = pd.read_csv(ORTHO_CACHE)
print(f"Loaded ortholog table: {len(ortho_df)} rows; columns: {list(ortho_df.columns)}")
if "orthology_type" in ortho_df.columns:
    ortho_df = ortho_df[ortho_df["orthology_type"] == "ortholog_one2one"].copy()
    print(f"  one2one only: {len(ortho_df)} rows")

ensg2musg = {}
for _, row in ortho_df.iterrows():
    h = strip_ensembl_gene_id(str(row["human_ensembl_id"]))
    m = strip_ensembl_gene_id(str(row["mouse_ensembl_id"]))
    if h and m:
        ensg2musg[h] = m
print(f"  human ENSG -> mouse ENSMUSG: {len(ensg2musg)} pairs")

musg2col = {}
for vi in range(bcg.n_vars):
    m = strip_ensembl_gene_id(str(bcg.var["gene_ids"].iloc[vi]))
    if m and m not in musg2col:
        musg2col[m] = vi
print(f"  ENSMUSG with a BCG column (first col wins): {len(musg2col)} / {bcg.n_vars}")

ensg_to_bcg_col = {}
for ensg, musg in ensg2musg.items():
    ci = musg2col.get(musg)
    if ci is not None:
        ensg_to_bcg_col[ensg] = ci
print(f"  human ENSG with a BCG column: {len(ensg_to_bcg_col)}")


Loaded ortholog table: 14451 rows; columns: ['human_ensembl_id', 'human_gene_name', 'mouse_ensembl_id', 'mouse_gene_name', 'orthology_type']
  one2one only: 14451 rows
  human ENSG -> mouse ENSMUSG: 14451 pairs
  ENSMUSG with a BCG column (first col wins): 10866 / 10866
  human ENSG with a BCG column: 9827


## 3. Load gene axes (var_names only — do not load `.X`)

v08 IID Pearson and v08 train_test Pearson share this file. mixHVG is the sibling
flavor of the same model set; we count it but do not export that matrix here.


In [4]:
def h5ad_var_names(path):
    with h5py.File(path, "r") as f:
        g = f["var"]
        key = g.attrs.get("_index", "index")
        key = key.decode() if isinstance(key, bytes) else key
        return [v.decode() if isinstance(v, bytes) else str(v) for v in g[key][:]]


v08_genes = [strip_ensembl_gene_id(g) for g in h5ad_var_names(V08_PEARSON)]
v07_genes = [strip_ensembl_gene_id(g) for g in h5ad_var_names(V07_PEARSON)]
mix_genes = [strip_ensembl_gene_id(g) for g in h5ad_var_names(V08_MIXHVG)]

print(f"v08 Pearson: {len(v08_genes)}  sample {v08_genes[:5]}")
print(f"v07 Pearson: {len(v07_genes)}  sample {v07_genes[:5]}")
print(f"v08 mixHVG:  {len(mix_genes)}  sample {mix_genes[:5]}")
print(f"v08 Pearson ∩ v07 Pearson (axis overlap, not BCG): {len(set(v08_genes) & set(v07_genes))} / 1000")
print(f"v08 Pearson ∩ v08 mixHVG  (axis overlap, not BCG): {len(set(v08_genes) & set(mix_genes))} / 1000")


v08 Pearson: 1000  sample ['ENSG00000164047', 'ENSG00000143546', 'ENSG00000111341', 'ENSG00000163631', 'ENSG00000170323']
v07 Pearson: 1000  sample ['ENSG00000168484', 'ENSG00000143546', 'ENSG00000011465', 'ENSG00000196569', 'ENSG00000111341']
v08 mixHVG:  1000  sample ['ENSG00000204287', 'ENSG00000108849', 'ENSG00000019582', 'ENSG00000232629', 'ENSG00000168484']
v08 Pearson ∩ v07 Pearson (axis overlap, not BCG): 721 / 1000
v08 Pearson ∩ v08 mixHVG  (axis overlap, not BCG): 674 / 1000


## 4. Shared-gene count (the number notebook 16 reported as 582 / 1000 on v07 Pearson)

A gene is **shared** when the v08 Pearson ENSG has a one-to-one mouse ortholog
**and** that ENSMUSG is a column in BCG. Missing genes are split into
"no one-to-one ortholog" vs "ortholog exists but not measured in BCG".


In [5]:
def coverage_row(label, genes):
    n = len(genes)
    has_ortho = [g in ensg2musg for g in genes]
    present = [g in ensg_to_bcg_col for g in genes]
    n_ortho = int(sum(has_ortho))
    n_present = int(sum(present))
    n_ortho_missing_in_bcg = int(sum(
        g in ensg2musg and g not in ensg_to_bcg_col for g in genes
    ))
    n_no_ortho = n - n_ortho
    return {
        "axis": label,
        "n_axis": n,
        "n_shared_with_bcg": n_present,
        "coverage_pct": round(100.0 * n_present / n, 1),
        "n_one2one_ortholog": n_ortho,
        "n_no_one2one_ortholog": n_no_ortho,
        "n_ortho_but_absent_in_bcg": n_ortho_missing_in_bcg,
    }


coverage_df = pd.DataFrame([
    coverage_row("v08_iid Pearson (export target)", v08_genes),
    coverage_row("v07 atlas-full Pearson (notebook 16)", v07_genes),
    coverage_row("v08 mixHVG (sibling flavor; not exported)", mix_genes),
])
coverage_df.to_csv(COVERAGE_CSV, index=False)
print(coverage_df.to_string(index=False))
print(f"\nwrote {COVERAGE_CSV}")

v08_shared = int(coverage_df.loc[0, "n_shared_with_bcg"])
print(
    f"\nHeadline: BCG shares {v08_shared} / {len(v08_genes)} genes "
    f"({100 * v08_shared / len(v08_genes):.1f}%) with the v08 IID Pearson axis."
)


                                     axis  n_axis  n_shared_with_bcg  coverage_pct  n_one2one_ortholog  n_no_one2one_ortholog  n_ortho_but_absent_in_bcg
          v08_iid Pearson (export target)    1000                507          50.7                1000                      0                        493
     v07 atlas-full Pearson (notebook 16)    1000                582          58.2                1000                      0                        418
v08 mixHVG (sibling flavor; not exported)    1000                650          65.0                1000                      0                        350

wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_mouse_outputs/bcg_v08_iid_pearson_hvg_coverage.csv

Headline: BCG shares 507 / 1000 genes (50.7%) with the v08 IID Pearson axis.


## 5. Project mentor `.X` onto the v08 Pearson axis and write

Output is **(n_cells, 1000)** in v08 Pearson order. Missing genes are zeros.
Filename is tagged `_uncapped_v08_iid` so it cannot collide with notebook 16.


In [6]:
n_target = len(v08_genes)
bcg_cols = [ensg_to_bcg_col.get(g, -1) for g in v08_genes]
n_present = sum(1 for c in bcg_cols if c >= 0)
print(f"projecting BCG .X onto {n_target} v08 Pearson genes; present={n_present}")

X_bcg = bcg.X.toarray() if sp_sparse.issparse(bcg.X) else np.asarray(bcg.X, dtype=np.float32)
X_new = np.zeros((bcg.n_obs, n_target), dtype=np.float32)
for j, c in enumerate(bcg_cols):
    if c >= 0:
        X_new[:, j] = X_bcg[:, c]

new_var = pd.DataFrame(index=pd.Index(v08_genes, name="ensg"))
new_var["in_bcg"] = [c >= 0 for c in bcg_cols]
keep_cols = [c for c in ["condition", "species", "cell_type", "study", "_scvi_batch"] if c in bcg.obs.columns]
aligned = ad.AnnData(
    X=X_new,
    obs=bcg.obs[keep_cols].copy(),
    var=new_var,
)
aligned.uns["gene_axis"] = "hvg_pearson_residuals_a_uncapped_v08"
aligned.uns["model_set"] = "uncapped_v08_iid"
aligned.uns["n_shared_with_bcg"] = int(n_present)
aligned.write_h5ad(ALIGNED_OUT)
print(
    f"wrote {ALIGNED_OUT}: shape {aligned.shape}, "
    f".X mean={float(aligned.X.mean()):.4f}, max={float(aligned.X.max()):.4f}"
)


projecting BCG .X onto 1000 v08 Pearson genes; present=507
wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_pearson_residuals_uncapped_v08_iid.h5ad: shape (1406, 1000), .X mean=0.3475, max=19.8487


## 6. Which v08 Pearson genes are missing from BCG?


In [7]:
rows = []
for j, ensg in enumerate(v08_genes):
    musg = ensg2musg.get(ensg)
    if ensg in ensg_to_bcg_col:
        continue
    rows.append({
        "ensg": ensg,
        "axis_index": j,
        "mouse_ensembl_id": musg if musg else "",
        "reason": "ortholog_absent_in_bcg" if musg else "no_one2one_ortholog",
    })
missing_df = pd.DataFrame(rows)
missing_df.to_csv(MISSING_CSV, index=False)
print(missing_df["reason"].value_counts().to_string())
print(f"\n{len(missing_df)} missing genes → {MISSING_CSV}")
print(missing_df.head(15).to_string(index=False))


reason
ortholog_absent_in_bcg    493

493 missing genes → /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_mouse_outputs/bcg_missing_hvg_v08_iid_pearson.csv
           ensg  axis_index   mouse_ensembl_id                 reason
ENSG00000164047           0 ENSMUSG00000038357 ortholog_absent_in_bcg
ENSG00000111341           2 ENSMUSG00000030218 ortholog_absent_in_bcg
ENSG00000163631           3 ENSMUSG00000029368 ortholog_absent_in_bcg
ENSG00000170323           4 ENSMUSG00000062515 ortholog_absent_in_bcg
ENSG00000170891           5 ENSMUSG00000062329 ortholog_absent_in_bcg
ENSG00000188257           6 ENSMUSG00000058908 ortholog_absent_in_bcg
ENSG00000168484           7 ENSMUSG00000022097 ortholog_absent_in_bcg
ENSG00000011465           8 ENSMUSG00000019929 ortholog_absent_in_bcg
ENSG00000090382           9 ENSMUSG00000020177 ortholog_absent_in_bcg
ENSG00000196754          12 ENSMUSG00000094018 ortholog_absent_in_bcg
ENSG00000139219          13 ENSMUSG00000022